# 07b · Ablación por familias, segundo algoritmo (PPO) y simulación visual

Parte de GC de la guía 07 (pregunta 7) y la simulación visual de la guía 06. Lee los artefactos que escribe
`python main.py`: `ablation.csv`, `algorithm_comparison.csv`, `validation_results_ppo.csv` y `agent_simulation.gif`.
Aquí no se entrena nada. La ablación y PPO **se reportan**: no se usan para reelegir el modelo que fue a prueba.

In [1]:
import sys
sys.path.insert(0, "..")
sys.modules.pop("code", None)  # IPython ya importó el módulo `code` estándar (ver docs/decisiones.md)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from code import data, train
from main import ARTIFACTS

ablation = pd.read_csv(ARTIFACTS / "ablation.csv", dtype={"fold": str})
comparison = pd.read_csv(ARTIFACTS / "algorithm_comparison.csv", dtype={"fold": str})
ppo = pd.read_csv(ARTIFACTS / "validation_results_ppo.csv")

## 1. Ablación por familias

Se reentrena la **configuración elegida** de cada fold (cfg 8, con sus pasos y sus 3 semillas) quitando una familia de
la observación a la vez; `cuenta` pone en 0 las 3 variables de la cuenta (caída desde el máximo, última operación y
horas plano). La variante `ninguna` reproduce el modelo elegido (el pipeline lo verifica contra
`validation_results_fold_{k}.csv`). Cada celda es la mediana entre semillas; `delta_*` es la diferencia contra
`ninguna` (negativo = quitar la familia empeora = la familia aporta).

In [2]:
pd.DataFrame(data.FEATURE_FAMILIES.items(), columns=["familia", "variables"]).set_index("familia")

,variables
familia,
returns,"[ret_1h, ret_4h, ret_24h, ret_168h]"
volatility,"[rv_24h, rv_168h, parkinson_24h, vol_ratio_24_..."
trend,"[ema_gap_24h, ema_gap_168h, ema_gap_720h, draw..."
volume,"[volume_z_168h, volume_ratio_24_168]"
calendar,"[hour_sin, hour_cos, dow_sin, dow_cos]"
belief,"[belief_drift, belief_drift_se, belief_drift_t..."


In [3]:
ablation.pivot_table(index="variant", columns="fold", values=["delta_val", "delta_test"], sort=False).round(3)

delta_val                             delta_test                \
fold               1      2      3      4  media          1      2      3   
variant                                                                     
ninguna        0.000  0.000  0.000  0.000  0.000      0.000  0.000  0.000   
returns       -0.274 -0.062 -0.317 -0.106 -0.190      0.273  0.270  0.132   
volatility    -0.497  0.042 -0.167 -0.447 -0.267     -0.061  0.175 -0.072   
trend         -0.470 -0.183 -0.551  0.075 -0.282      0.112  0.416  0.396   
volume        -0.399  0.030 -0.123 -0.092 -0.146      0.071  0.370  0.178   
calendar      -0.417 -0.273 -0.484 -0.060 -0.309      0.249  0.259 -0.054   
belief        -0.560  0.148 -0.176 -0.096 -0.171      0.010  0.099  0.219   
cuenta        -0.967 -0.150  0.005 -0.269 -0.345     -0.378  0.028  0.139   

                          
fold            4  media  
variant                   
ninguna     0.000  0.000  
returns     0.090  0.191  
volatility  0.045  0.022  
trend      -0.130  0.198  
volume     -0.028  0.148  
calendar    0.112  0.142  
belief      0.117  0.111  
cuenta     -0.185 -0.099

**Cuidado con la validación.** La base es el mejor de 96 candidatos en validación (configuración × pasos), elegido
por su mediana; las variantes no pasaron por esa selección. Por eso `delta_val` está sesgado hacia lo negativo (la
maldición del ganador): que todas las familias "aporten" en validación es en parte un artefacto de la selección. La
prueba no pasó por ninguna selección, así que es la comparación más honesta, aunque con 3 semillas y 4 folds es ruidosa.

In [4]:
variants = [v for v in train.ABLATION_VARIANTS if v != "ninguna"]
per_fold = ablation[ablation["fold"] != "media"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, col, title in zip(axes, ["delta_val", "delta_test"], ["validación", "prueba"]):
    mean = ablation[ablation["fold"] == "media"].set_index("variant").loc[variants, col]
    ax.barh(variants, mean, color=["#257de0" if v != "cuenta" else "#f47230" for v in variants], alpha=0.8)
    for v in variants:
        ax.scatter(per_fold[per_fold["variant"] == v][col], [v] * 4, color="k", s=10, zorder=3)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_title(f"Δ crecimiento log al quitar la familia: {title} (barra = media, puntos = folds)")
axes[0].invert_yaxis()
fig.tight_layout()

In [5]:
g = per_fold[per_fold["variant"] != "ninguna"].groupby("variant", sort=False)
pd.DataFrame({"folds donde empeora (validación)": g.apply(lambda d: int((d["delta_val"] < 0).sum())),
              "folds donde empeora (prueba)": g.apply(lambda d: int((d["delta_test"] < 0).sum())),
              "operaciones en prueba (media)": g["test_n_trades"].mean()}).loc[variants]

,folds donde empeora (validación),folds donde empeora (prueba),operaciones en prueba (media)
variant,,,
returns,4,0,97.25
volatility,3,2,110.25
trend,3,1,76.25
volume,3,1,85.50
calendar,4,1,141.25
belief,3,0,81.75
cuenta,3,2,176.75


**Lectura.**

- Ninguna familia de mercado tiene un aporte que se sostenga fuera de la selección: en prueba, quitar rendimientos,
  creencia, tendencia, volumen o calendario **no empeora** en 3 o 4 de los 4 folds. Cuadra con la parte 02, donde la
  señal de dirección es débil y frágil a la fase de muestreo.
- La **cuenta** es la única variante que empeora en validación y, en media, también en prueba, y con ella el agente
  opera bastante más. Las variables de la cuenta funcionan como freno: después de perder, el agente espera. Es el
  mismo mecanismo que explica la fragilidad del fold 2 en la sensibilidad (parte 07, JME).
- Las semillas pesan tanto como las familias: en la base, el crecimiento log en prueba del fold 1 va de −0.15 a −0.42
  según la semilla.

## 2. Segundo algoritmo: PPO

PPO de SB3 (on-policy, actor-crítico) con la misma búsqueda en validación: rejilla de 8 (tasa de aprendizaje
{1e-4, 3e-4} × γ {0.99, 0.995} × coeficiente de entropía {0, 0.01}, red [64, 64]), 3 semillas, puntos de control cada
50k pasos y el mismo criterio A. **Aproximación declarada:** el GAE de PPO descuenta con γ por decisión, no con γ^τ.

In [6]:
cols = ["config_id", "timesteps", "seed", "val_median", "val_min", "val_max", "candidates_above_zero",
        "test_log_growth", "test_sharpe", "test_n_trades"]
comparison.set_index(["fold", "algo"])[cols].round(3)

config_id  timesteps  seed  val_median  val_min  val_max  \
fold  algo                                                             
1     dqn         8.0   300000.0   0.0       0.225   -0.255    0.609   
      ppo         6.0   250000.0   2.0      -0.414   -1.894   -0.284   
2     dqn         8.0   300000.0   1.0      -0.253   -0.485   -0.125   
      ppo         0.0   300000.0   2.0      -0.318   -0.490   -0.076   
3     dqn         8.0   250000.0   1.0       0.005   -0.998    0.037   
      ppo         0.0   100000.0   2.0      -0.353   -0.989    0.169   
4     dqn         8.0   300000.0   1.0      -0.063   -0.568    0.053   
      ppo         2.0   300000.0   0.0      -0.159   -0.531   -0.143   
media dqn         NaN        NaN   NaN      -0.022   -0.576    0.143   
      ppo         NaN        NaN   NaN      -0.311   -0.976   -0.083   

            candidates_above_zero  test_log_growth  test_sharpe  test_n_trades  
fold  algo                                                                      
1     dqn                    2.00           -0.151       -1.213          33.00  
      ppo                    0.00           -0.158       -0.227         518.00  
2     dqn                    0.00           -0.294       -1.038         142.00  
      ppo                    0.00           -0.712       -2.112         211.00  
3     dqn                    1.00           -0.207       -0.572         186.00  
      ppo                    0.00           -0.924       -3.422         249.00  
4     dqn                    0.00           -0.029       -0.516          18.00  
      ppo                    0.00           -0.156       -2.767          27.00  
media dqn                    0.75           -0.170       -0.835          94.75  
      ppo                    0.00           -0.488       -2.132         251.25

**Lectura.** PPO queda por debajo de DQN en validación y en prueba, y opera más. Ningún candidato de PPO tiene
mediana positiva en validación en ningún fold. Que el algoritmo off-policy con el descuento semi-Markov correcto sea el
mejor de los dos es consistente con la formulación; pero ninguno de los dos le gana a no operar después de costos.

In [7]:
medians = ppo.groupby(["fold", "config_id", "timesteps"])["val_log_growth"].median()
medians.groupby("fold").describe()[["min", "50%", "max"]].round(3)

,min,50%,max
fold,,,
1,-1.854,-1.241,-0.414
2,-1.120,-0.816,-0.318
3,-1.596,-0.822,-0.353
4,-1.852,-0.856,-0.159


## 3. Simulación visual (`agent_simulation.gif`)

El tramo sale de un criterio fijo, las 2 semanas de prueba con más operaciones del agente entre los 4 folds (fold 3,
23 de febrero a 9 de marzo de 2025). La animación solo lee la bitácora y las curvas de capital de la evaluación:
precio barra por barra, entradas (▲ largo, ▼ corto), salidas con su razón (stop, take-profit u horizonte), niveles de
stop y take-profit mientras la posición está abierta y el capital del agente contra comprar y mantener BTC.

![simulación](../artifacts/agent_simulation.gif)